# elbo-loss-sum-with-beta — worked example 1: Beta-weighted ELBO from two scalar terms

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `elbo-loss-sum-with-beta`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

The beta-VAE training loss is `reconstruction + beta * kl`. `beta` is a scalar hyperparameter that trades reconstruction sharpness against latent-space regularity: `beta = 0` ignores the prior entirely, larger `beta` pulls the posterior harder toward `N(0, I)`. Both loss terms are already reduced to scalars before they are combined.

## Worked solution

We are given a scalar reconstruction loss and a scalar KL loss and must form the composite training objective.

1. `recon` is a 0-D tensor — the per-batch mean squared (or BCE) reconstruction error. `kl` is a 0-D tensor — the per-batch mean Gaussian KL versus the standard normal prior.
2. The composite loss is the straight line `recon + beta * kl`. The slope in `beta` is exactly `kl`, the intercept is `recon`.
3. We compute it for a single `beta`, then print the value to confirm it is a scalar. Because both inputs are 0-D tensors and `beta` is a Python float, the result stays a 0-D tensor — ready to `.backward()`.
4. We sanity-check that at `beta = 0` the loss equals `recon` alone, which is the degenerate plain-autoencoder case.

In [ ]:
import torch as t

t.manual_seed(0)
recon = t.tensor(2.5)
kl = t.tensor(0.8)

def elbo_loss(recon, kl, beta):
    return recon + beta * kl

loss = elbo_loss(recon, kl, 1.0)
print('loss:', float(loss))
print('beta=0 -> recon:', float(elbo_loss(recon, kl, 0.0)) == float(recon))